### Loss Functions

#### 0. Loss vs metric

A loss function must be differentiable (or subdifferentiable), it drives training through gradients. A metric does not need to be, it just needs to be interpretable for humans (accuracy, F1). Sometimes the same formula serves both roles, cross-entropy is trained on directly and also reported as log loss. Sometimes they diverge on purpose, accuracy is not differentiable so it can never be a loss function directly, cross-entropy is used as a smooth proxy for it during training.

Toy regression setup used below: 3 predictions.
```
actual:    [10, 20, 30]
predicted: [12, 18, 45]
errors:    [-2, 2, -15]
```
Note the third point is a big outlier error (-15), used to show how differently each loss reacts to it.


#### 1. MSE (L2 loss)

Formula: L = mean(error^2). Gradient w.r.t. prediction: dL/dpred = -2*error/n (for each point, ignoring the mean's 1/n for a single point: dL/dpred = -2*error).

Worked example, errors [-2, 2, -15]:
```
squared errors: [4, 4, 225]
MSE = (4+4+225)/3 = 233/3 = 77.67
```
The outlier (error=-15) contributes 225 of the total 233, over 96% of the loss from one point. Gradient for that point: -2*(-15) = 30, a huge gradient, the optimizer gets pulled hard toward fixing this one point over the other two.

#### 2. MAE (L1 loss)

Formula: L = mean(|error|). Gradient: dL/dpred = -sign(error), constant magnitude regardless of error size. Not differentiable exactly at error=0 (subgradient used there in practice, usually 0).

Worked example, same errors:
```
|errors|: [2, 2, 15]
MAE = (2+2+15)/3 = 19/3 = 6.33
```
Gradient magnitude is the same 1 (well, -1 or +1) for the outlier as for the small errors. MAE does not let one bad point dominate the gradient the way MSE does, but it also does not push especially hard to fix it either, same pull regardless of how wrong.


In [ ]:
import numpy as np

actual = np.array([10, 20, 30])
predicted = np.array([12, 18, 45])
errors = actual - predicted

mse = np.mean(errors ** 2)
mae = np.mean(np.abs(errors))
mse_gradients = -2 * errors
mae_gradients = -np.sign(errors)

print("errors:", errors)
print("MSE:", mse, "| per-point gradients:", mse_gradients)
print("MAE:", mae, "| per-point gradients:", mae_gradients)
print("\noutlier's share of MSE:", (errors[2]**2) / np.sum(errors**2))

#### 3. Huber loss

Quadratic near zero (like MSE), linear past a threshold delta (like MAE). Smooth everywhere (unlike MAE), bounded outlier influence (unlike MSE).

Formula: L(e) = 0.5*e^2 if |e|<=delta, else delta*(|e|-0.5*delta). Gradient: dL/de = e if |e|<=delta, else delta*sign(e).

Worked example, delta=3, same errors [-2, 2, -15]:
```
e=-2 (|e|<=3): loss = 0.5*4 = 2, gradient = -2
e=2  (|e|<=3): loss = 0.5*4 = 2, gradient = 2
e=-15 (|e|>3): loss = 3*(15-1.5) = 40.5, gradient = 3*sign(-15) = -3
```
Compare the outlier's gradient across all three losses now: MSE gives it -30 (dominates everything), MAE gives it -1 (same as every other point), Huber gives it -3 (bigger than the small-error points' gradients of -2 and 2, but capped, not 30). Huber is a middle ground on purpose.


In [ ]:
delta = 3

def huber_loss(e, delta):
    return np.where(np.abs(e) <= delta, 0.5 * e**2, delta * (np.abs(e) - 0.5 * delta))

def huber_gradient(e, delta):
    return np.where(np.abs(e) <= delta, e, delta * np.sign(e))

print("Huber loss per point:", huber_loss(errors, delta))
print("Huber gradient per point:", huber_gradient(errors, delta))

print("\ngradient comparison for the outlier (error=-15):")
print("MSE gradient:", mse_gradients[2])
print("MAE gradient:", mae_gradients[2])
print("Huber gradient:", huber_gradient(errors, delta)[2])

#### Entropy, the foundation cross-entropy builds on

Information = surprise. Rain in a rainforest carries ~0 information (expected, not surprising). Rain in the Sahara carries a lot of information (very surprising).

Entropy = the average surprise/uncertainty in a distribution. An always-heads coin has entropy 0, you are never surprised by the outcome. A fair coin has maximum entropy, maximally uncertain before the flip.

Cross-entropy = the EXTRA surprise from using a wrong belief Q to predict the real distribution P: cross-entropy = entropy(P) + KL(P||Q) (see section 8 below for KL divergence itself). If Q=P exactly, cross-entropy equals entropy, no extra surprise from being wrong, because you are not wrong. If Q differs from P, cross-entropy exceeds entropy by exactly the KL divergence term.

Why this matters for classification specifically: hard labels (a one-hot true class, 100% one way) have entropy exactly 0, there is no uncertainty in the ground truth itself. So cross-entropy = 0 + KL(P||Q) = KL(P||Q) = the error itself, not error plus some irreducible baseline noise. Predicting p=0.7 for a true label of 1.0, the cross-entropy value IS exactly the penalty from that 0.3 gap, nothing else is mixed in. This is why cross-entropy makes such a clean training loss, it is not measuring "error plus inherent uncertainty", for hard-labeled classification it is measuring error, full stop.

#### 4. Binary cross-entropy (log loss)

Formula: L = -[y*log(p) + (1-y)*log(1-p)]. Gradient w.r.t. the raw score z (before sigmoid): dL/dz = p-y. This clean form is exactly why logistic regression and gradient boosting both use p-y as their gradient, cross-entropy's derivative through sigmoid simplifies to this.

Classification toy: doc1, true y=1, predicted p=0.9 (confident and correct). doc2, true y=0, predicted p=0.7 (confident and WRONG).
```
loss(doc1) = -[1*log(0.9) + 0*log(0.1)] = -log(0.9) = 0.105
loss(doc2) = -[0*log(0.7) + 1*log(0.3)] = -log(0.3) = 1.204
```
doc2's loss is over 11x doc1's, despite both predictions being equally "confident" (0.9 vs 0.7, not wildly different). Confident and wrong is punished far more than confident and right is rewarded, log's shape does that automatically, it blows up as p approaches 0 for the true class.

#### 5. Categorical cross-entropy

Multi-class extension, same idea per class: L = -sum(y_k * log(p_k)), y one-hot. Gradient: dL/dz_k = p_k - y_k, same clean form as binary, applied per class. Already the exact formula used in the logreg and XGBoost multi-class notes.


In [ ]:
def binary_cross_entropy(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

loss_doc1 = binary_cross_entropy(1, 0.9)
loss_doc2 = binary_cross_entropy(0, 0.7)

print("loss(doc1), confident and correct:", loss_doc1)
print("loss(doc2), confident and wrong:", loss_doc2)
print("ratio:", loss_doc2 / loss_doc1)

#### 6. Hinge loss (SVM)

Formula: L = max(0, 1 - y*z), y in {-1, +1} (not {0,1}, SVM's convention), z = raw model score (not a probability). Gradient: -y if y*z < 1, else 0.

Not just "wrong gets penalized", hinge penalizes anything that is not confidently correct by a margin of at least 1. A correct prediction that is barely correct still gets loss.

Worked example, y=+1 (true positive class) for 3 different raw scores z:
```
z=2.0 (confidently correct, margin=2*1=2 > 1):    loss = max(0, 1-2.0) = 0
z=0.5 (correct but not confident, margin=0.5 < 1): loss = max(0, 1-0.5) = 0.5
z=-1.0 (wrong):                                     loss = max(0, 1-(-1.0)) = 2.0
```
z=0.5 predicts the right side (positive) but still gets penalized, because it is inside the margin. This margin-based pressure, not just "get the sign right", is the mechanic that makes SVM look for the maximally separating boundary rather than just any separating one.

#### 7. Focal loss (class imbalance)

Formula: L = -(1-p_t)^gamma * log(p_t), where p_t is the model's predicted probability for the TRUE class (p if y=1, 1-p if y=0). gamma is a tunable focusing parameter, typically 2.

Standard cross-entropy treats every example the same regardless of how easy it already is. Focal loss adds the (1-p_t)^gamma factor, which shrinks toward 0 as p_t approaches 1 (already easy, well-classified examples), so their contribution to the total loss gets down-weighted, letting hard/rare examples dominate training more.

Worked example, gamma=2, two examples both correctly classified but at different confidence:
```
easy example, p_t=0.95: focal_loss = (1-0.95)^2 * -log(0.95) = 0.0025 * 0.051 = 0.000128
hard example, p_t=0.55: focal_loss = (1-0.55)^2 * -log(0.55) = 0.2025 * 0.598 = 0.121
```
Compare to plain cross-entropy for the same two: -log(0.95)=0.051 and -log(0.55)=0.598, a ratio of about 12x. Under focal loss the ratio becomes 0.121/0.000128 which is about 945x, the easy example's contribution gets crushed far more aggressively, forcing training to focus on the hard, often minority-class examples. Directly relevant to something like the fraud project's rare-typology classes.


In [ ]:
def hinge_loss(y, z):
    return max(0, 1 - y * z)

for z in [2.0, 0.5, -1.0]:
    print(f"z={z}: hinge loss = {hinge_loss(1, z)}")

def focal_loss(p_t, gamma=2):
    return -((1 - p_t) ** gamma) * np.log(p_t)

def cross_entropy_single(p_t):
    return -np.log(p_t)

easy, hard = 0.95, 0.55
print("\neasy example (p_t=0.95): CE =", cross_entropy_single(easy), "| focal =", focal_loss(easy))
print("hard example (p_t=0.55): CE =", cross_entropy_single(hard), "| focal =", focal_loss(hard))
print("CE ratio hard/easy:", cross_entropy_single(hard) / cross_entropy_single(easy))
print("focal ratio hard/easy:", focal_loss(hard) / focal_loss(easy))

#### 8. KL divergence

Formula: KL(P||Q) = sum(P(x) * log(P(x)/Q(x))). Measures how one probability distribution Q differs from a reference distribution P. Not symmetric, KL(P||Q) is not equal to KL(Q||P) in general.

Worked example, true distribution P (a one-hot label, class 2 is correct) vs model's predicted distribution Q, over 3 classes:
```
P = [0, 1, 0]
Q = [0.2, 0.7, 0.1]

KL(P||Q) = 0*log(0/0.2) + 1*log(1/0.7) + 0*log(0/0.1)
```
The 0*log(0/x) terms are defined as 0 by convention (limit as P(x) approaches 0). Only the true class's term survives:
```
KL(P||Q) = log(1/0.7) = log(1.4286) = 0.357
```
Compare to cross-entropy for the same setup: CE = -log(0.7) = 0.357. Identical here. When the true distribution is one-hot (the normal classification setup), KL divergence and cross-entropy differ only by a constant (the entropy of P, which is 0 for a one-hot distribution), so minimizing one minimizes the other. KL divergence matters as its own loss specifically when P is NOT one-hot, soft labels, knowledge distillation (student model matching a teacher model's full output distribution, not just its top prediction), or comparing two learned distributions directly.


In [ ]:
def kl_divergence(p, q):
    p, q = np.array(p), np.array(q)
    mask = p > 0  # skip 0*log(0/x) terms, defined as 0
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

P = [0, 1, 0]
Q = [0.2, 0.7, 0.1]

kl = kl_divergence(P, Q)
ce_single = -np.log(0.7)

print("KL(P||Q):", kl)
print("cross-entropy for the same setup:", ce_single)
print("equal when P is one-hot:", np.isclose(kl, ce_single))

#### 9. When to use which

- MSE: default regression loss, use when large errors are genuinely worse and outliers are real signal, not noise.
- MAE: regression with outliers that should not dominate training, or when the target has heavy-tailed noise.
- Huber: wants MSE's smoothness near zero but does not want a handful of outliers to hijack training, common middle-ground choice.
- Binary/categorical cross-entropy: default classification loss, almost always the right starting point.
- Hinge: SVM specifically, or any max-margin classifier, not used with probability outputs.
- Focal: classification with severe class imbalance, object detection (where background/easy negatives vastly outnumber the objects of interest) is where it was introduced, but applies anywhere rare classes get drowned out by easy majority-class examples.
- KL divergence: soft-label targets, knowledge distillation, variational methods (VAEs), anywhere the target itself is a distribution rather than a single true class.
